# Data preparation (Colab) — already run

This notebook built everything in `data/processed/`, `data/qvh_frames*/` and `data/qvh_thumbnails/` (QVHighlights parsing, YouTube downloads, frame extraction, eval-pair construction). It is kept as the record of how the dataset was made. **You do not need to run it** to use the project. It was written for Google Colab + Drive and downloads videos with yt-dlp.


## Setup

Run this first. It configures paths for Colab **or** local automatically.

In [ ]:
# === Unified setup — runs on BOTH Google Colab and local ===
# All data paths come from one source of truth (src/utils/paths.py). On Colab
# this mounts Drive, extracts the M1 feature tar to local SSD, and clones the
# upstream moment_detr repo; locally it just wires sys.path + resolved paths.
# Safe to re-run.
import os, sys, subprocess, time, tarfile, shutil
from pathlib import Path

def _find_repo_root():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "__init__.py").exists():
            return cand
    drive_repo = Path("/content/drive/MyDrive/thumbnail_project/thumbnail-consistency")
    return drive_repo if drive_repo.exists() else Path.cwd()

REPO_DIR = _find_repo_root()
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from types import SimpleNamespace
_on_colab = Path("/content").is_dir()
if _on_colab and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive; drive.mount("/content/drive")
_root = Path("/content/drive/MyDrive/thumbnail_project") if _on_colab else REPO_DIR / "data"
P = SimpleNamespace(on_colab=_on_colab, data_root=_root,
                    feature_dir=_root / "features" / "clip_features",
                    text_feature_dir=_root / "features" / "clip_text_features")
DRIVE_ROOT = P.data_root              # back-compat aliases for downstream cells
PROJECT_ROOT = P.data_root

if P.on_colab:
    for d in ["processed", "outputs", "qvh_videos", "qvh_frames", "qvh_thumbnails",
              "qvh_frames_within_neg", "qvh_frames_multi", "checkpoints"]:
        src, dst = DRIVE_ROOT / d, REPO_DIR / d
        if src.exists() and not (dst.exists() or dst.is_symlink()):
            try:
                os.symlink(src, dst)
            except OSError as e:
                print(f"  WARN symlink {d}: {e}")

    LOCAL_OUT = Path("/content/features_local")
    def _find_feats(root):
        clip = text = None
        for r, dirs, _ in os.walk(root):
            for d in dirs:
                p = Path(r) / d
                if d == "clip_features" and len(list(p.glob("*.npz"))) > 1000:
                    clip = p
                if d == "clip_text_features" and len(list(p.glob("*.npz"))) > 1000:
                    text = p
        return clip, text

    CLIP_FEAT_DIR = TEXT_FEAT_DIR = None
    if LOCAL_OUT.exists():
        CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feats(LOCAL_OUT)
    if not (CLIP_FEAT_DIR and TEXT_FEAT_DIR):
        archive = DRIVE_ROOT / "moment_detr_features.tar.gz"
        assert archive.exists(), f"Feature archive missing: {archive}"
        if LOCAL_OUT.exists():
            shutil.rmtree(LOCAL_OUT)
        LOCAL_OUT.mkdir(parents=True)
        print(f"[features] extracting {archive.stat().st_size/1e9:.2f} GB -> {LOCAL_OUT}")
        t0 = time.time()
        with tarfile.open(archive) as tf:
            tf.extractall(LOCAL_OUT)
        print(f"[features] extracted in {time.time()-t0:.0f}s")
        CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feats(LOCAL_OUT)

    MD_DIR = Path("/content/moment_detr_upstream")
    if not MD_DIR.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/jayleicn/moment_detr.git", str(MD_DIR)],
                       check=True)
    if str(MD_DIR) not in sys.path:
        sys.path.insert(0, str(MD_DIR))
else:
    CLIP_FEAT_DIR = P.feature_dir
    TEXT_FEAT_DIR = P.text_feature_dir
    md = REPO_DIR / "Moment_DETR" / "moment_detr"
    if md.exists() and str(md) not in sys.path:
        sys.path.insert(0, str(md))

try:
    import torch
    _dev = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    _dev = "unknown (torch not installed)"
print(f"Colab           : {P.on_colab}")
print(f"Repo            : {REPO_DIR}")
print(f"Device          : {_dev}")
print(f"clip_features   : {CLIP_FEAT_DIR}")
print(f"clip_text_feats : {TEXT_FEAT_DIR}")
print("Setup done.")


---

# Data collection & preparation

*(consolidated from `final_data_collection_preparation.ipynb`)*

# QVHighlights — Data Collection & Preparation
## Dataset for M1 (Moment Retrieval) and M2 (Thumbnail → Query)

---

### What this notebook does
1. Mounts Drive and sets up the project directory
2. Clones the moment_detr repo (annotations included)
3. Parses and explains every field in the JSONL annotation format
4. Groups segments by original YouTube video
5. Builds unified processed dataset for M1 and M2
6. Downloads a subset of raw video clips (for M2 frame extraction)
7. Extracts frames from downloaded clips
8. Constructs `(frame_image, query)` training pairs for M2
9. Saves final processed splits to Drive

**Design decisions are documented inline with explicit reasoning.**

# 1. Environment Setup & Drive Mount

In [ ]:
# Install required packages
!pip install -q yt-dlp opencv-python-headless Pillow tqdm
!pip install -q git+https://github.com/openai/CLIP.git

# ⚠️ RESET CELL — Run this ONLY when starting fresh
Delete all previously generated outputs: extracted frames, processed JSONL files,
and visualisation PNGs. Videos are preserved (expensive to re-download).

**Skip this cell if you are resuming a previous run.**


In [ ]:
# ── FULL RESET — deletes all generated outputs, keeps downloaded videos ──────
# Set CONFIRM = True to actually run the deletion.
CONFIRM = False #← change to True when you want to reset

if not CONFIRM:
    print("Reset skipped. Set CONFIRM = True to delete outputs.")
else:
    import shutil, os
    deleted = {"frames": 0, "processed": 0, "pngs": 0}

    # 1. Delete all extracted frames (.jpg)
    if FRAMES_DIR.exists():
        for f in FRAMES_DIR.glob("*.jpg"):
            f.unlink()
            deleted["frames"] += 1
        print(f"  Deleted {deleted['frames']} frames from {FRAMES_DIR}")

    # 2. Delete processed JSONL files
    if PROCESSED_DIR.exists():
        for f in PROCESSED_DIR.glob("*.jsonl"):
            f.unlink()
            deleted["processed"] += 1
        for f in PROCESSED_DIR.glob("*.json"):
            f.unlink()
            deleted["processed"] += 1
        print(f"  Deleted {deleted['processed']} processed files from {PROCESSED_DIR}")

    # 3. Delete visualisation PNGs
    if PROCESSED_DIR.exists():
        for f in PROCESSED_DIR.glob("*.png"):
            f.unlink()
            deleted["pngs"] += 1
        print(f"  Deleted {deleted['pngs']} PNG visualisations")

    print(f"\n✓ Reset complete. Videos in {VIDEO_DIR} are untouched.")
    print("You can now run the notebook from the beginning.")


# 2. Clone moment_detr Repository

The annotations (JSONL files) are included in the repo under `data/`.
No separate download is needed for the annotation files.

In [ ]:
if not (REPO_DIR / "data" / "highlight_train_release.jsonl").exists():
    print("Cloning moment_detr repo...")
    subprocess.run(
        ["git", "clone", "https://github.com/jayleicn/moment_detr.git", str(REPO_DIR)],
        check=True
    )
    print("Clone complete.")
else:
    print("Repo already cloned. Skipping.")

# Verify annotation files exist
for split in ["train", "val", "test"]:
    path = DATA_DIR / f"highlight_{split}_release.jsonl"
    exists = "✓" if path.exists() else "✗ MISSING"
    print(f"  {exists}  {path.name}")

# 3. Parse Annotation Files — Field-by-Field Explanation

Each line in the JSONL file is one `(video_segment, query)` pair.

| Field | Type | Meaning |
|---|---|---|
| `qid` | int | Unique query ID across the entire dataset |
| `query` | str | Human-written natural language description of a visual moment |
| `duration` | int | Length of the video segment in seconds (always ~150s) |
| `vid` | str | `{youtube_id}_{start_sec}_{end_sec}` — identifies which YouTube video AND which 150s segment |
| `relevant_windows` | list[list] | `[[start, end], ...]` — timestamps within the 150s segment where the query is answered |
| `relevant_clip_ids` | list[int] | Indices of 2-second clips (clip 0 = seconds 0–2, clip 1 = seconds 2–4, ...) |
| `saliency_scores` | list[list] | One sublist per `relevant_clip_id`. Each sublist = `[score_annotator1, score_annotator2, score_annotator3]`. Score 4 = Very Good, 0 = Very Bad |

**Key insight for M2:** The clip with the highest mean saliency score is the best visual representative of the query — this is our training frame for the thumbnail-to-query model.

In [ ]:
# [PIPELINE_PATCH_v1]
def load_jsonl(path):
    """Load JSONL with explicit utf-8 (B1)."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

# Load all three splits
train_raw = load_jsonl(DATA_DIR / "highlight_train_release.jsonl")
val_raw   = load_jsonl(DATA_DIR / "highlight_val_release.jsonl")
test_raw  = load_jsonl(DATA_DIR / "highlight_test_release.jsonl")

print(f"Train samples : {len(train_raw):,}")
print(f"Val samples   : {len(val_raw):,}")
print(f"Test samples  : {len(test_raw):,}")
print(f"Total         : {len(train_raw)+len(val_raw)+len(test_raw):,}")


In [ ]:
# [PIPELINE_PATCH_v3_principled]
# Principled best-frame selection with a 3-tier tie-breaking hierarchy.
#
# Replaces v2's "middle of tied list" heuristic. The hierarchy:
#
#   Tier 1  unique_max
#           Exactly one clip has the highest mean saliency. Use it.
#
#   Tier 2  annotator_agreement
#           Multiple clips tied on mean. Pick the one with the LOWEST
#           per-annotator variance -- i.e. the clip the 3 annotators agreed on
#           most strongly. High agreement = stronger supervision signal.
#
#   Tier 3  temporal_position
#           Still tied (zero-variance agreement on multiple clips). Prefer the
#           LATER clip. Heuristic: in QVH queries describing actions, the
#           climax tends to sit at the end of the relevant window.
#
#   (Optional)  sharpness
#           If `frame_paths` is supplied (i.e. frames are pre-extracted for
#           every tied candidate), use Laplacian variance as a focus measure
#           and pick the sharpest. Off by default in V1 due to chicken-and-egg
#           with extraction.
#
# Returns: (best_clip_id, best_timestamp_seconds, selection_reason).

def laplacian_sharpness(frame_path):
    """Variance of the Laplacian -- standard cheap focus measure.
    Higher = sharper. Returns 0.0 on read failure (so it never wins a tie)."""
    import cv2
    img = cv2.imread(str(frame_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.0
    return float(cv2.Laplacian(img, cv2.CV_64F).var())


def select_best_frame(relevant_clip_ids, saliency_scores,
                      relevant_windows=None, frame_paths=None):
    """3-tier hierarchical selection. See docstring above for tiers."""
    if not relevant_clip_ids or not saliency_scores:
        return None, None, "no_annotation"

    mean_scores  = [sum(s) / 3 for s in saliency_scores]
    max_score    = max(mean_scores)
    tied_indices = [i for i, s in enumerate(mean_scores) if s == max_score]

    # ---- Tier 1: unique max ------------------------------------------------
    if len(tied_indices) == 1:
        best_local = tied_indices[0]
        reason = "unique_max"

    else:
        # ---- Tier 2: annotator agreement (lowest variance) -----------------
        variances = [
            sum((x - mean_scores[i]) ** 2 for x in saliency_scores[i]) / 3
            for i in tied_indices
        ]
        min_var = min(variances)
        agreement_winners = [
            tied_indices[j] for j, v in enumerate(variances) if v == min_var
        ]

        if len(agreement_winners) == 1:
            best_local = agreement_winners[0]
            reason = "annotator_agreement"

        else:
            # ---- Tier 3a: sharpness (only if frame_paths given) ----------
            if frame_paths is not None:
                sharps = [laplacian_sharpness(frame_paths[i])
                          for i in agreement_winners]
                best_local = agreement_winners[sharps.index(max(sharps))]
                reason = "sharpness"
            # ---- Tier 3b: temporal position (later clip) -----------------
            else:
                best_local = max(agreement_winners,
                                 key=lambda i: relevant_clip_ids[i])
                reason = "temporal_position"

    best_clip_id   = relevant_clip_ids[best_local]
    best_timestamp = best_clip_id * 2 + 1.0   # centre of the 2-second clip
    return best_clip_id, best_timestamp, reason


In [ ]:
# ── Annotated example — print and explain one training record ────────────────
sample = train_raw[0]
print("=== RAW ANNOTATION RECORD ===")
print(json.dumps(sample, indent=2))

print("\n=== FIELD BREAKDOWN ===")
print(f"  qid            : {sample['qid']} — unique query identifier")
print(f"  vid            : {sample['vid']}")

# Parse vid into components
parts = sample["vid"].rsplit("_", 2)
yt_id, seg_start, seg_end = parts[0], float(parts[1]), float(parts[2])
print(f"    youtube_id   : {yt_id}")
print(f"    seg_start    : {seg_start}s  (where this 150s segment starts in the full video)")
print(f"    seg_end      : {seg_end}s")
print(f"    YouTube URL  : https://www.youtube.com/watch?v={yt_id}&t={int(seg_start)}")

print(f"  query          : \"{sample['query']}\"")
print(f"  duration       : {sample['duration']}s  (length of this segment)")
print(f"  relevant_windows: {sample['relevant_windows']}  (seconds within the 150s segment)")
print(f"  size of relevant_windows:{len(sample['relevant_windows'])} ")
print(f"  relevant_clip_ids: {sample['relevant_clip_ids']}  (2-sec clip indices)")
print(f"  size of relevent_clip_ids: {len(sample['relevant_clip_ids'])} ")
print(f"\n  saliency_scores: {sample['saliency_scores']}")
print(f"  size of saliency_scores: {len(sample['saliency_scores'])} ")
print(f"    → {len(sample['saliency_scores'])} clips × 3 annotators each")
mean_scores = [sum(s) / 3 for s in sample["saliency_scores"]]
best_clip_id, best_timestamp, selection_reason = select_best_frame(
    relevant_clip_ids=sample["relevant_clip_ids"],
    saliency_scores=sample["saliency_scores"],
)
print(f"    → mean scores per clip : {[round(s,2) for s in mean_scores]}")
print(f"    → best clip id         : {best_clip_id} (selection: {selection_reason})")
print(f"    → best frame timestamp : {best_timestamp}s  ← this is the M2 training frame")

# 4. Design Decisions

## 4.1 Use official JSONL as-is or create a processed version?

**Decision: Create a processed version.**

Reasons:
- The official JSONL has fields we do not need (raw saliency_scores as nested lists)
- We need to add derived fields: `youtube_id`, `seg_start`, `seg_end`, `best_clip_id`, `best_frame_timestamp`, `mean_saliency`
- We need to track which samples have downloaded video frames (for M2)
- A processed version with a clear schema makes the training code cleaner and less error-prone

We keep the original JSONL untouched and write new files: `train_processed.jsonl`, `val_processed.jsonl`, `test_processed.jsonl`

---

## 4.2 How to group segments from the same original YouTube video?

Each `vid` is `{youtube_id}_{start}_{end}`. The same YouTube video can appear multiple times with different start/end windows and different queries. For M1 training, this is fine — each segment is independent. For analysis, grouping by `youtube_id` lets us understand how many unique source videos we have.

---

## 4.3 Subset size — is 2000–2500 training samples reasonable?

**Decision: Yes, use 2000 training + full val + full test.**

Justification:
- The full train set has ~7,000 samples. Training M1 (Moment-DETR) on the full set takes ~12 hours on RTX 2080Ti. On Colab T4, that is 20–30 hours — too slow for iterative experimentation.
- 2000 samples = ~28% of train. The paper shows Moment-DETR with 5.4K in-domain pretraining videos achieves strong results. At 2000 we expect measurable performance with reasonable training time (~6–8 hours on T4).
- We use the official val and test splits without modification to ensure our metrics are comparable to the paper's reported numbers.
- We sample from train by **YouTube channel diversity** — not random — to avoid having multiple segments from the same video dominate the subset.

**Split strategy: stratified by video category (daily vlog / travel vlog / news) to match the full dataset's distribution.**

# 5. Build Processed Dataset

In [ ]:
# [PIPELINE_PATCH_v3_principled]
# process_record() now delegates clip selection to select_best_frame() so the
# 3-tier hierarchy is the single source of truth. Also stores selection_reason
# on each record for downstream auditing (e.g. "what fraction of training pairs
# came from a unique-max clip vs a tied-agreement clip?").

def process_record(item, has_saliency=True):
    vid = item["vid"]
    parts = vid.rsplit("_", 2)
    yt_id, seg_start, seg_end = parts[0], float(parts[1]), float(parts[2])

    processed = {
        "qid"              : item["qid"],
        "query"            : item["query"],
        "duration"         : item["duration"],
        "vid"              : vid,
        "youtube_id"       : yt_id,
        "seg_start"        : seg_start,
        "seg_end"          : seg_end,
        "relevant_windows" : item.get("relevant_windows", []),
        "relevant_clip_ids": item.get("relevant_clip_ids", []),
        "video_downloaded" : False,
        "frame_path"       : None,
    }

    if has_saliency and item.get("saliency_scores") and item.get("relevant_clip_ids"):
        clip_ids = item["relevant_clip_ids"]
        scores   = item["saliency_scores"]
        best_cid, best_ts, reason = select_best_frame(
            relevant_clip_ids=clip_ids,
            saliency_scores=scores,
            relevant_windows=item.get("relevant_windows"),
        )
        mean_scores = [sum(s) / 3 for s in scores]
        processed["best_clip_id"]      = best_cid
        processed["best_timestamp"]    = best_ts
        processed["mean_saliency"]     = round(max(mean_scores), 3)
        processed["selection_reason"]  = reason
    else:
        processed["best_clip_id"]     = None
        processed["best_timestamp"]   = None
        processed["mean_saliency"]    = None
        processed["selection_reason"] = "no_annotation"

    return processed


# Process all splits
train_proc = [process_record(r, has_saliency=True)  for r in train_raw]
val_proc   = [process_record(r, has_saliency=True)  for r in val_raw]
test_proc  = [process_record(r, has_saliency=False) for r in test_raw]

print(f"Processed -> train: {len(train_proc)}, val: {len(val_proc)}, test: {len(test_proc)}")

# Audit the tie-breaking hierarchy on the train split.
from collections import Counter
reason_counts = Counter(r.get("selection_reason", "no_annotation") for r in train_proc)
print("\nSelection reasons (train split):")
for k, v in reason_counts.most_common():
    print(f"  {k:25s}: {v:5d}  ({100*v/len(train_proc):5.1f}%)")

print("\nSample processed record:")
print(json.dumps(train_proc[0], indent=2))


# 6. Subset Selection — Stratified by Category

In [ ]:
# [PIPELINE_PATCH_v1]
# Category inference with explicit vlog bucket + "other" fallback (B2).
# Avoids over-counting daily_vlog when no keyword matches.

TRAVEL_KEYWORDS = {"beach","hotel","tour","plane","swim","visit","travel","ocean",
                   "mountain","temple","city","airport","restaurant","street","market",
                   "trip","resort","cruise","sightseeing","tourist"}
NEWS_KEYWORDS   = {"protest","police","report","interview","press","fire","flood",
                   "hurricane","crowd","rally","march","officer","military",
                   "election","government","news","reporter","anchor"}
VLOG_KEYWORDS   = {"camera","selfie","vlog","talks","shows","explains","unboxing",
                   "kitchen","cooking","makeup","family","baby","dog","cat","bedroom"}

def infer_category(query):
    words = set(query.lower().split())
    if words & NEWS_KEYWORDS:   return "news"
    if words & TRAVEL_KEYWORDS: return "travel_vlog"
    if words & VLOG_KEYWORDS:   return "daily_vlog"
    return "other"

for r in train_proc:
    r["category"] = infer_category(r["query"])

from collections import Counter
cat_counts = Counter(r["category"] for r in train_proc)
print("Category distribution in full train set:")
for cat, cnt in sorted(cat_counts.items()):
    print(f"  {cat:15s}: {cnt:5d}  ({cnt/len(train_proc)*100:.1f}%)")

# Persist mapping qid -> category for downstream eval.
cat_map = {r["qid"]: r["category"] for r in train_proc}
with open(PROCESSED_DIR / "category_map.json", "w", encoding="utf-8") as f:
    json.dump(cat_map, f)
print(f"Saved category_map.json ({len(cat_map)} entries)")


In [ ]:
# ── Stratified subset: 2000 samples matching category proportions ────────────
SUBSET_SIZE = 4000
random.seed(42)  # reproducibility

by_cat = defaultdict(list)
for r in train_proc:
    by_cat[r["category"]].append(r)

subset = []
for cat, records in by_cat.items():
    proportion = len(records) / len(train_proc)
    n = round(SUBSET_SIZE * proportion)
    sampled = random.sample(records, min(n, len(records)))
    subset.append((cat, sampled))
    print(f"  {cat:15s}: {len(sampled):4d} samples ({proportion*100:.1f}%)")

train_subset = []
for _, s in subset:
    train_subset.extend(s)

# Shuffle to mix categories
random.shuffle(train_subset)
print(f"\nFinal subset size: {len(train_subset)}")
print(f"Val size (full): {len(val_proc)}")
print(f"Test size (full): {len(test_proc)}")

# 7. Save Processed Splits

In [ ]:
# [PIPELINE_PATCH_v1]
def save_jsonl(records, path):
    """Save JSONL with explicit utf-8 + non-ASCII safe (B1)."""
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(records):,} records -> {path}")

save_jsonl(train_subset, PROCESSED_DIR / "train_processed.jsonl")
save_jsonl(val_proc,     PROCESSED_DIR / "val_processed.jsonl")
save_jsonl(test_proc,    PROCESSED_DIR / "test_processed.jsonl")
save_jsonl(train_proc,   PROCESSED_DIR / "train_full_processed.jsonl")

print("\nProcessed files saved to Drive.")


# 8. Download Video Clips for M2 Training

We download only the 150-second segment per video, not the full YouTube video.
Using `--download-sections` in yt-dlp limits each download to the exact window.

**Resolution: 480p** — sufficient for frame extraction, reduces storage ~4× vs 1080p.

**Estimated storage:** ~30–50 MB per clip × 2000 clips ≈ 60–100 GB total.
For a first run, we download the subset (2000 clips).
If storage is limited, use the `MAX_DOWNLOADS` parameter to start with fewer.

In [ ]:
# [PIPELINE_PATCH_v1]
# yt-dlp downloader with multi-client fallback + stderr logging (B3).

MAX_DOWNLOADS    = 4000   # set lower for a quick test run
PARALLEL_WORKERS = 6

YT_CLIENTS = ["android", "web", "ios"]  # try these in order on failure

def _run_ytdlp(cmd, timeout):
    p = subprocess.run(cmd, timeout=timeout, capture_output=True, text=True)
    return p.returncode, p.stdout, p.stderr

def download_clip(record):
    vid      = record["vid"]
    yt_id    = record["youtube_id"]
    start    = record["seg_start"]
    end      = record["seg_end"]
    out_path = VIDEO_DIR / f"{vid}.mp4"

    if out_path.exists() and out_path.stat().st_size > 10_000:
        record["video_downloaded"] = True
        return {"vid": vid, "status": "skip", "path": str(out_path), "err": None}

    url = f"https://www.youtube.com/watch?v={yt_id}"
    last_err = None
    for client in YT_CLIENTS:
        cmd = [
            "yt-dlp",
            "--download-sections", f"*{start}-{end}",
            "--format", "bestvideo[height<=480][ext=mp4]+bestaudio[ext=m4a]/best[height<=480]",
            "--merge-output-format", "mp4",
            "--extractor-args", f"youtube:player_client={client}",
            "--quiet", "--no-warnings",
            "-o", str(out_path),
            url,
        ]
        try:
            rc, _, err = _run_ytdlp(cmd, timeout=180)
            if rc == 0 and out_path.exists():
                record["video_downloaded"] = True
                return {"vid": vid, "status": "ok", "path": str(out_path),
                        "err": None, "client": client}
            last_err = (err or "").strip()[-300:]
        except subprocess.TimeoutExpired:
            last_err = "timeout"
            continue
    record["video_downloaded"] = False
    status = "timeout" if last_err == "timeout" else "fail"
    return {"vid": vid, "status": status, "path": None, "err": last_err}

print(f"Downloading up to {MAX_DOWNLOADS} clips with {PARALLEL_WORKERS} workers...")

to_download = train_subset[:MAX_DOWNLOADS]
results = {"ok": 0, "skip": 0, "fail": 0, "timeout": 0}
download_log = []

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    futures = {executor.submit(download_clip, r): r for r in to_download}
    for i, future in enumerate(as_completed(futures), 1):
        result = future.result()
        results[result["status"]] = results.get(result["status"], 0) + 1
        download_log.append(result)
        if i % 50 == 0 or i == len(to_download):
            print(f"  [{i:4d}/{len(to_download)}]  "
                  f"ok={results['ok']}  skip={results['skip']}  "
                  f"fail={results['fail']}  timeout={results['timeout']}")

with open(PROCESSED_DIR / "download_log.json", "w", encoding="utf-8") as f:
    json.dump(download_log, f, indent=2)

print(f"\nDownload complete: ok={results['ok']}, skip={results['skip']}, "
      f"fail={results['fail']}, timeout={results['timeout']}")
print("Failure samples (last 5):")
for r in [r for r in download_log if r["status"] in ("fail", "timeout")][:5]:
    print(f"  {r['vid']}: {r['err']}")


# 9. Extract Best Frame for Each Training Sample (M2)

For each downloaded clip we:
1. Sample `N_CANDIDATES=10` frames uniformly across the 2-second best clip.
2. Score every candidate against the query with **CLIP ViT-B/32** cosine similarity.
3. Save the highest-scoring frame as the M2 training image.

The `clip_score` field is stored per record — it acts as a semantic quality signal for downstream filtering (low-score pairs are likely noisy supervision).

GPU calls are serialised with a threading lock so video I/O remains parallel while CLIP inference is protected from race conditions.

### Pre-extraction fix — clear corrupted zero-score frames
Frames that previously got `clip_score = 0.0` due to the CLIP thread race condition
are re-extracted by deleting their `.jpg` files before running `extract_frame`.
A genuine CLIP cosine similarity is never exactly `0.0`, so any `0.0` in the
saved JSONL is a confirmed bug artifact.

This cell is safe to re-run — it only deletes frames with `clip_score == 0.0`.


In [ ]:
# [PIPELINE_PATCH_v2_center]
# This cell was previously used to delete clip_score == 0.0 frames caused by a
# CLIP race condition. With v2 (center-frame, no CLIP) that race no longer
# exists, so this cell is now a no-op. Left in place so cell ordering stays
# stable for the v1 -> v2 migration.
print("v2 pipeline: no race-condition cleanup needed.")


In [ ]:
# [PIPELINE_PATCH_v2_center]
# Center-frame extraction. No CLIP, no race condition, deterministic.
#
# Replaces the CLIP-ranked extractor: empirically the CLIP-ranking step added
# little visible quality but introduced a race condition, GPU memory churn, and
# a tunable score threshold. Switching back to the simple center-of-best-clip
# strategy gives equivalent supervision signal at ~10x the throughput.
#
# Side effects (set on the record):
#   frame_path     : str | absent
#   best_timestamp : seconds (centre of the 2-second best clip)
#   clip_score     : None  (kept for schema compatibility downstream)
#   video_downloaded: bool

import cv2
from PIL import Image
from pathlib import Path


def extract_frame(record):
    vid        = record["vid"]
    best_cid   = record.get("best_clip_id")
    video_path = VIDEO_DIR / f"{vid}.mp4"
    frame_out  = FRAMES_DIR / f"{vid}.jpg"

    # Annotated as "no relevant clip" (e.g. test split) -> nothing to extract.
    if best_cid is None:
        record["video_downloaded"] = video_path.exists()
        record["clip_score"] = None
        return record

    if not video_path.exists():
        record["video_downloaded"] = False
        record["clip_score"] = None
        return record

    # Cached: frame already on disk.
    if frame_out.exists():
        record["video_downloaded"] = True
        record["frame_path"]       = str(frame_out)
        record["clip_score"]       = None
        record["best_timestamp"]   = best_cid * 2 + 1.0
        return record

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        record["video_downloaded"] = False
        record["clip_score"] = None
        return record

    # Seek to centre of the 2-second clip via MSEC (VFR-safe).
    center_ms = best_cid * 2_000 + 1_000
    cap.set(cv2.CAP_PROP_POS_MSEC, float(center_ms))
    ret, frame = cap.read()
    cap.release()

    if not ret or frame is None:
        record["video_downloaded"] = False
        record["clip_score"] = None
        return record

    pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    pil.resize((224, 224), Image.LANCZOS).save(str(frame_out), quality=92)

    record["video_downloaded"] = True
    record["frame_path"]       = str(frame_out)
    record["clip_score"]       = None
    record["best_timestamp"]   = best_cid * 2 + 1.0
    return record


print("Extracting center frames (no CLIP)...")
updated_records = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(extract_frame, r): r for r in train_subset}
    for i, future in enumerate(as_completed(futures), 1):
        updated_records.append(future.result())
        if i % 200 == 0 or i == len(train_subset):
            done = sum(1 for r in updated_records if r.get("frame_path"))
            print(f"  [{i:4d}/{len(train_subset)}]  frames extracted: {done}")

updated_records.sort(key=lambda x: x["qid"])
train_subset = updated_records

n_with_frame = sum(1 for r in train_subset if r.get("frame_path"))
print(f"\nFrames extracted: {n_with_frame} / {len(train_subset)}")


# 10. Build M2 Training Pairs & Save Final Dataset

In [ ]:
# ── M2 training pairs: only records where frame was extracted ─────────────────
m2_pairs = [
    {
        "qid"           : r["qid"],
        "frame_path"    : r["frame_path"],
        "query"         : r["query"],
        "vid"           : r["vid"],
        "best_timestamp": r["best_timestamp"],
        "mean_saliency" : r["mean_saliency"],
        "clip_score"    : r.get("clip_score"),    # None = fallback/unscored
    }
    for r in train_subset if r.get("frame_path")
]

print(f"M2 training pairs available: {len(m2_pairs)}")
print("\nSample M2 pair:")
print(json.dumps(m2_pairs[0], indent=2))

In [ ]:
# ── Save final processed files ────────────────────────────────────────────────
save_jsonl(train_subset, PROCESSED_DIR / "train_processed.jsonl")    # updated with frame_path
save_jsonl(m2_pairs,     PROCESSED_DIR / "m2_train_pairs.jsonl")     # M2 only
save_jsonl(val_proc,     PROCESSED_DIR / "val_processed.jsonl")
save_jsonl(test_proc,    PROCESSED_DIR / "test_processed.jsonl")

print("\nAll processed files saved.")
print("\nFinal dataset summary:")
print(f"  M1 train (subset)       : {len(train_subset):,} samples")
print(f"  M1 val (full official)  : {len(val_proc):,} samples")
print(f"  M1 test (full official) : {len(test_proc):,} samples")
print(f"  M2 (frame,query) pairs  : {len(m2_pairs):,} pairs")

# 11. Dataset Statistics & Validation

In [ ]:
import matplotlib.pyplot as plt

# ── 1. Saliency score distribution ──────────────────────────────────────────
saliency_vals = [r["mean_saliency"] for r in train_subset if r["mean_saliency"] is not None]

# ── 2. Query length distribution ────────────────────────────────────────────
query_lengths = [len(r["query"].split()) for r in train_subset]

# ── 3. Relevant window duration distribution ─────────────────────────────────
window_durations = []
for r in train_subset:
    for w in r["relevant_windows"]:
        window_durations.append(w[1] - w[0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Dataset Statistics — Training Subset (2000 samples)", fontsize=13)

axes[0].hist(saliency_vals, bins=20, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].set_title("Best Clip Mean Saliency Score\n(higher = better M2 training frame)")
axes[0].set_xlabel("Mean Saliency (0–4 scale)")
axes[0].axvline(np.mean(saliency_vals), color="red", linestyle="--",
                label=f"mean={np.mean(saliency_vals):.2f}")
axes[0].legend()

axes[1].hist(query_lengths, bins=20, color="darkorange", edgecolor="white", alpha=0.85)
axes[1].set_title("Query Length Distribution\n(words per query)")
axes[1].set_xlabel("Number of words")
axes[1].axvline(np.mean(query_lengths), color="red", linestyle="--",
                label=f"mean={np.mean(query_lengths):.1f}")
axes[1].legend()

axes[2].hist(window_durations, bins=20, color="green", edgecolor="white", alpha=0.85)
axes[2].set_title("Relevant Window Duration\n(seconds)")
axes[2].set_xlabel("Duration (seconds)")
axes[2].axvline(np.mean(window_durations), color="red", linestyle="--",
                label=f"mean={np.mean(window_durations):.1f}s")
axes[2].legend()

plt.tight_layout()
plt.savefig(str(PROCESSED_DIR / "dataset_stats.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean saliency score  : {np.mean(saliency_vals):.3f} / 4.0")
print(f"Mean query length    : {np.mean(query_lengths):.1f} words")
print(f"Mean window duration : {np.mean(window_durations):.1f} seconds")

In [ ]:
# ── Validate all required fields are present ─────────────────────────────────
required_m1 = {"qid","query","vid","youtube_id","seg_start","seg_end",
                "relevant_windows","relevant_clip_ids","duration"}
required_m2 = {"qid","frame_path","query","vid","best_timestamp","mean_saliency","clip_score"}

def validate_records(records, required_fields, name):
    issues = []
    for i, r in enumerate(records):
        missing = required_fields - set(r.keys())
        if missing:
            issues.append(f"  Record {i} (qid={r.get('qid','?')}) missing: {missing}")
    if issues:
        print(f"\n{name} — {len(issues)} issues found:")
        for issue in issues[:5]:
            print(issue)
    else:
        print(f"  {name} — ✓ all {len(records)} records valid")

print("Validation:")
validate_records(train_subset, required_m1, "M1 train")
validate_records(m2_pairs,     required_m2, "M2 pairs")
validate_records(val_proc,     required_m1 - {"relevant_clip_ids"}, "M1 val")

# Visualize Visualizee!!!

In [ ]:
# random.seed(82)
# ==============================
# PATHS
# ==============================
PROCESSED_DIR = Path("/content/drive/MyDrive/thumbnail_project/processed")
VIDEO_DIR     = Path("/content/drive/MyDrive/thumbnail_project/qvh_videos")

# ==============================
# LOAD DATA
# ==============================
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

m2_pairs = load_jsonl(PROCESSED_DIR / "m2_train_pairs.jsonl")

print(f"Loaded {len(m2_pairs)} M2 samples")

# ==============================
# 1. SHOW RANDOM FRAMES + QUERIES
# ==============================
import textwrap

def show_random_samples(n=10):
    samples = random.sample(m2_pairs, min(n, len(m2_pairs)))

    fig, axes = plt.subplots(2, 5, figsize=(15, 6))

    axes = axes.flatten()

    for i, s in enumerate(samples):
        img = Image.open(s["frame_path"])

        axes[i].imshow(img)
        axes[i].axis("off")

        title = s["query"]
        wrapped = "\n".join(textwrap.wrap(title, width=25))

        axes[i].set_title(wrapped, fontsize=8)


    plt.tight_layout(pad=0.6)
    plt.subplots_adjust(top=0.88)

    plt.suptitle("Random (Frame, Query) Samples", fontsize=14)

    plt.show()

def show_nice_sample(sample):
    import matplotlib.pyplot as plt
    import cv2
    from PIL import Image
    import textwrap

    video_path = VIDEO_DIR / f"{sample['vid']}.mp4"

    if not video_path.exists():
        print("Video not found")
        return

    # =========================
    # LOAD BEST FRAME
    # =========================
    img = Image.open(sample["frame_path"])

    # =========================
    # LOAD VIDEO CONTEXT
    # =========================
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    best_frame = int(sample["best_timestamp"] * fps)

    offsets = [-8, -4, 0, 4, 8]
    frames = []

    for offset in offsets:
        target = best_frame + offset
        cap.set(cv2.CAP_PROP_POS_FRAMES, target)
        ret, frame = cap.read()

        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        else:
            frames.append(None)

    cap.release()

    # =========================
    # FIGURE LAYOUT
    # =========================
    fig = plt.figure(figsize=(12, 6))

    # --- BEST FRAME ---
    ax_main = plt.subplot2grid((2, 5), (0, 1), colspan=3)
    ax_main.imshow(img)
    ax_main.axis("off")
    ax_main.set_title("BEST FRAME", fontsize=11, weight="bold")

    # --- QUERY ---
    query = "\n".join(textwrap.wrap(sample["query"], width=60))
    fig.text(0.5, 0.48, query, ha="center", fontsize=11)

    # --- CONTEXT FRAMES ---
    for i, f in enumerate(frames):
        ax = plt.subplot2grid((2, 5), (1, i))

        if f is not None:
            ax.imshow(f)
        else:
            ax.text(0.5, 0.5, "N/A", ha='center', va='center')

        ax.axis("off")

        if offsets[i] == 0:
            ax.set_title("BEST", color="red", fontsize=10, weight="bold")
        else:
            ax.set_title(f"{offsets[i]}", fontsize=9)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.3)
    plt.show()
# ==============================
# RUN EVERYTHING
# ==============================

# 1. random visualization
show_random_samples(10)

In [ ]:
# random.seed(87)
# 2. detailed example
sample = random.choice(m2_pairs)
# show_detailed_sample(sample)

# 3. video + frame validation
show_nice_sample(sample)

## DEBUG

In [ ]:
def debug_one_video(sample):
    import matplotlib.pyplot as plt
    from PIL import Image
    import textwrap

    def get_base_video_id(vid):
        return vid.rsplit("_", 2)[0]

    base_id = get_base_video_id(sample["vid"])

    related = [
        s for s in m2_pairs
        if get_base_video_id(s["vid"]) == base_id
    ]

    print(f"Base video: {base_id}")
    print(f"Found {len(related)} segments")

    # if only one segment
    if len(related) == 1:
        s = related[0]
        img = Image.open(s["frame_path"])

        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.axis("off")

        q = "\n".join(textwrap.wrap(s["query"], width=40))

        plt.title(q, fontsize=11, pad=10)
        plt.tight_layout()
        plt.show()
        return

    # if more than one
    cols = min(5, len(related))
    rows = (len(related) + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = axes.flatten()

    for i, s in enumerate(related):
        img = Image.open(s["frame_path"])

        axes[i].imshow(img)
        axes[i].axis("off")

        q = "\n".join(textwrap.wrap(s["query"], width=30))

        start = s["vid"].split("_")[-2]
        end   = s["vid"].split("_")[-1]

        axes[i].set_title(f"{start}-{end}\n{q}", fontsize=8)

    for j in range(i+1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout(pad=0.5)
    plt.show()

sample = random.choice(m2_pairs)
debug_one_video(sample)

# 12. Final Directory Structure & What Goes Where

```
thumbnail_project/
moment_detr/
  data/
  highlight_train_release.jsonl   ← original, untouched
  highlight_val_release.jsonl
  highlight_test_release.jsonl
  subs_train.jsonl                ← ASR subtitles for M1 pretraining

processed/
  train_processed.jsonl             ← 2000 samples, enriched fields, frame_path
  train_full_processed.jsonl        ← all ~7000 train samples (no frames)
  val_processed.jsonl               ← official val, enriched
  test_processed.jsonl              ← official test, no saliency
  m2_train_pairs.jsonl              ← (frame_path, query) pairs for M2
  download_log.json                 ← which videos succeeded/failed
  dataset_stats.png                 ← statistics plot

qvh_videos/
  {vid}.mp4                         ← downloaded 150s clips
  ...

qvh_frames/
  {vid}.jpg                         ← extracted best frame (224×224) for M2
  ...
```

## Next steps
- **M1:** Load `train_processed.jsonl` + precomputed features → train Moment-DETR
- **M2:** Load `m2_train_pairs.jsonl` → train CLIP visual encoder + GPT-2 decoder
- **Evaluation:** Always use official `val_processed.jsonl` and `test_processed.jsonl`

**Limitation:**

* Modern video understanding systems face a dual challenge when generating textual queries from visual content. In our case, the **model is trained** on raw video **frames**, while in **real-world deployment it receives highly stylized thumbnails**. These thumbnails often include visual enhancements such as color adjustments, text overlays, zoomed-in faces, and other editing effects. This creates a domain gap between training data and inference inputs, which can negatively impact the model's ability to generalize.

(mafish kalam kefaya yawsaf (query) al thumbnial)
* At the same time, the system also suffers from **limitations in its language generation capabilities**. The model is trained on a relatively constrained set of queries, which restricts its vocabulary and expressive power. As a result, it may generate overly simple, repetitive, or even incoherent descriptions, especially when dealing with visually complex or unfamiliar thumbnails.

Combined Challenges
* **Domain Shift:**
A mismatch between clean training frames and real-world thumbnails leads to reduced performance.
* **Weak Visual-Text Alignment:**
Thumbnails may not perfectly represent the exact moment described by the query, introducing semantic noise.
* **Limited Vocabulary:**
The model struggles to describe diverse or complex scenes due to restricted language exposure.
* **Incoherent Generation:**
Outputs may be grammatically incorrect or semantically inconsistent, particularly under distribution shift.

# [PIPELINE_PATCH_v1]
# 13. Download Original YouTube Thumbnails (eval-time domain-gap probe)

These are **not** used for training M2. They are saved as metadata for evaluating
the domain gap between extracted frames and real stylized thumbnails (per
FINAL_DESIGN section 2, Case D-light).

Output: `qvh_thumbnails/{youtube_id}.jpg` and `thumbnail_index.jsonl`.


In [ ]:
# [PIPELINE_PATCH_v4_direct_thumbnails]
# Direct HTTP download from i.ytimg.com -- bypasses yt-dlp entirely.
# YouTube CDN thumbnails are public and don't trigger bot detection.
import urllib.request, urllib.error, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

THUMB_DIR = PROJECT_ROOT / "qvh_thumbnails"
THUMB_DIR.mkdir(parents=True, exist_ok=True)

# Standard YouTube thumbnail resolutions, tried in order of preference.
THUMB_URLS = [
    "https://i.ytimg.com/vi/{yt}/maxresdefault.jpg",   # 1280x720, may 404
    "https://i.ytimg.com/vi/{yt}/sddefault.jpg",       # 640x480
    "https://i.ytimg.com/vi/{yt}/hqdefault.jpg",       # 480x360, always exists
]
UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
      "AppleWebKit/537.36 (KHTML, like Gecko) "
      "Chrome/120.0 Safari/537.36")


def download_thumbnail(record):
    yt_id   = record["youtube_id"]
    out_jpg = THUMB_DIR / f"{yt_id}.jpg"
    if out_jpg.exists() and out_jpg.stat().st_size > 1000:
        return {"yt_id": yt_id, "status": "skip", "path": str(out_jpg)}

    last_err = None
    for tmpl in THUMB_URLS:
        url = tmpl.format(yt=yt_id)
        try:
            req = urllib.request.Request(url, headers={"User-Agent": UA})
            with urllib.request.urlopen(req, timeout=15) as resp:
                data = resp.read()
            if len(data) > 1000:                      # rule out the 1x1 placeholder
                out_jpg.write_bytes(data)
                return {"yt_id": yt_id, "status": "ok",
                        "path": str(out_jpg), "url": url, "size": len(data)}
        except urllib.error.HTTPError as e:
            last_err = f"HTTP {e.code}"
            continue                                  # 404 -> try lower resolution
        except (urllib.error.URLError, TimeoutError, OSError) as e:
            last_err = str(e)[:200]
            continue
    return {"yt_id": yt_id, "status": "fail", "err": last_err or "no_resolution_worked"}


# Deduplicate by youtube_id (one thumbnail per YT video).
unique_yt = list({r["youtube_id"]: r for r in train_subset}.values())
print(f"Downloading thumbnails for {len(unique_yt)} unique YouTube videos...")

thumb_results = []
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = {ex.submit(download_thumbnail, r): r for r in unique_yt}
    for i, fut in enumerate(as_completed(futures), 1):
        thumb_results.append(fut.result())
        if i % 100 == 0 or i == len(unique_yt):
            ok   = sum(1 for r in thumb_results if r["status"] in ("ok", "skip"))
            fail = sum(1 for r in thumb_results if r["status"] == "fail")
            print(f"  [{i:4d}/{len(unique_yt)}]  ok={ok}  fail={fail}")

# Per-resolution breakdown for the writeup.
from collections import Counter
res_used = Counter(r.get("url", "").rsplit("/", 1)[-1].replace(".jpg", "")
                   for r in thumb_results if r["status"] == "ok")
print(f"\nResolutions used: {dict(res_used)}")

# Build thumbnail_index.jsonl keyed by qid.
index = []
for r in train_subset:
    p = THUMB_DIR / f"{r['youtube_id']}.jpg"
    if p.exists():
        index.append({"qid": r["qid"], "vid": r["vid"],
                      "youtube_id": r["youtube_id"],
                      "thumbnail_path": str(p)})

with open(PROCESSED_DIR / "thumbnail_index.jsonl", "w", encoding="utf-8") as f:
    for x in index:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")

# Save failure log for debugging if needed.
with open(PROCESSED_DIR / "thumbnail_download_log.json", "w", encoding="utf-8") as f:
    json.dump(thumb_results, f, indent=2)

print(f"\nThumbnails available for {len(index)} / {len(train_subset)} records.")
if any(r["status"] == "fail" for r in thumb_results):
    fails = [r for r in thumb_results if r["status"] == "fail"]
    print(f"Failures: {len(fails)} -- first 3:")
    for r in fails[:3]:
        print(f"  {r['yt_id']}: {r.get('err')}")

# [PIPELINE_PATCH_v1]
# 14. Build M1 Training Subset in Official QVHighlights Format

Moment-DETR's loader expects the **original** QVH schema (not our processed one).
We emit `m1_train_subset.jsonl` by filtering the official train file using the
qids in our 2000-sample subset. We also copy `m1_val.jsonl` verbatim.

This file is consumed directly by `train.py --train_path` if we ever retrain.
For V1 we use the official checkpoint, but this file is still needed for
feature-coverage audits.


In [ ]:
# [PIPELINE_PATCH_v1]
# Filter official train JSONL by our subset qids -> m1_train_subset.jsonl.
subset_qids = {r["qid"] for r in train_subset}
m1_train = [r for r in train_raw if r["qid"] in subset_qids]

save_jsonl(m1_train, PROCESSED_DIR / "m1_train_subset.jsonl")
save_jsonl(val_raw,  PROCESSED_DIR / "m1_val.jsonl")

print(f"M1 train subset: {len(m1_train)} records (official schema)")
print(f"M1 val full    : {len(val_raw)} records (official schema)")

# Feature-coverage audit (R3 mitigation in FINAL_DESIGN section 9).
# We expect the official feature dir layout under REPO_DIR/features/clip_features.
feat_dir = REPO_DIR / "features" / "clip_features"
if feat_dir.exists():
    have = {p.stem for p in feat_dir.glob("*.npz")}
    missing = [r["vid"] for r in m1_train if r["vid"] not in have]
    print(f"Feature files present: {len(have)}")
    print(f"Subset records missing features: {len(missing)}")
    if missing[:3]:
        print(f"  examples: {missing[:3]}")
else:
    print("(Feature dir not unpacked yet; run feature unpack before training audit.)")


# [PIPELINE_PATCH_v1]
# 15. Build M2 Validation Pairs (300 from official val)

M2 currently has no validation split. Without one, we cannot early-stop on
CLIP-text similarity. This cell extracts frames for ~300 val records using the
same `extract_frame` pipeline, producing `m2_val_pairs.jsonl`.

Held out separately so it never leaks into M2 training.


In [ ]:
# [PIPELINE_PATCH_v1] # [PIPELINE_PATCH_v7_selfcontained_m2_val]
# Self-contained M2 val pairs builder: defines its own downloader inline so a
# kernel restart between section 8 and section 15 cannot raise NameError.

import subprocess, json, random
from concurrent.futures import ThreadPoolExecutor, as_completed

random.seed(2026)

M2_VAL_N   = 300
YT_CLIENTS = ["android", "web", "ios"]


def _dl(record):
    """Download one 150-second clip if missing. Returns status dict."""
    vid      = record["vid"]
    out_path = VIDEO_DIR / f"{vid}.mp4"
    if out_path.exists() and out_path.stat().st_size > 10_000:
        record["video_downloaded"] = True
        return {"vid": vid, "status": "skip"}
    url = f"https://www.youtube.com/watch?v={record['youtube_id']}"
    last_err = None
    for client in YT_CLIENTS:
        cmd = ["yt-dlp",
               "--download-sections",
               f"*{record['seg_start']}-{record['seg_end']}",
               "--format",
               "bestvideo[height<=480][ext=mp4]+bestaudio[ext=m4a]/best[height<=480]",
               "--merge-output-format", "mp4",
               "--extractor-args", f"youtube:player_client={client}",
               "--quiet", "--no-warnings",
               "-o", str(out_path), url]
        try:
            p = subprocess.run(cmd, timeout=180, capture_output=True, text=True)
            if p.returncode == 0 and out_path.exists():
                record["video_downloaded"] = True
                return {"vid": vid, "status": "ok", "client": client}
            last_err = (p.stderr or "")[-200:]
        except subprocess.TimeoutExpired:
            last_err = "timeout"
    record["video_downloaded"] = False
    return {"vid": vid, "status": "fail", "err": last_err}


# Stratified pick by mean_saliency tertile
val_with_sal = [r for r in val_proc if r.get("mean_saliency") is not None]
val_with_sal.sort(key=lambda r: r["mean_saliency"])
t = len(val_with_sal) // 3
buckets = [val_with_sal[:t], val_with_sal[t:2*t], val_with_sal[2*t:]]
val_pick = []
for b in buckets:
    val_pick.extend(random.sample(b, min(M2_VAL_N // 3, len(b))))
print(f"Selected {len(val_pick)} val records for M2 evaluation pairs.")

print("Downloading val clips...")
val_dl_results = []
with ThreadPoolExecutor(max_workers=6) as ex:
    futures = {ex.submit(_dl, r): r for r in val_pick}
    for i, fut in enumerate(as_completed(futures), 1):
        val_dl_results.append(fut.result())
        if i % 50 == 0 or i == len(val_pick):
            ok = sum(1 for r in val_dl_results if r["status"] in ("ok", "skip"))
            print(f"  [{i:4d}/{len(val_pick)}]  ok={ok}")

print("\nExtracting M2 val frames...")
val_pick_extracted = []
with ThreadPoolExecutor(max_workers=6) as ex:
    futures = {ex.submit(extract_frame, r): r for r in val_pick}
    for i, fut in enumerate(as_completed(futures), 1):
        val_pick_extracted.append(fut.result())

m2_val_pairs = [
    {"qid": r["qid"], "vid": r["vid"], "frame_path": r["frame_path"],
     "query": r["query"], "best_timestamp": r["best_timestamp"],
     "mean_saliency": r["mean_saliency"], "clip_score": r.get("clip_score")}
    for r in val_pick_extracted if r.get("frame_path")
]

save_jsonl(m2_val_pairs, PROCESSED_DIR / "m2_val_pairs.jsonl")
print(f"\nM2 val pairs: {len(m2_val_pairs)} / {len(val_pick)} requested")


# [PIPELINE_PATCH_v1]
# 16. Build Consistency Evaluation Set (positives + hard negatives)

This is the **primary evaluation artifact** for the pipeline. Per FINAL_DESIGN
section 5 and the score in section 7:

- **Positive pair**: (frame_i, vid_i) -- the frame came from this video.
- **Hard negative**: (frame_i, vid_j) where i != j and the queries are
  semantically dissimilar (CLIP-text cosine < 0.6) so the negative isn't
  accidentally consistent.

Output: `consistency_eval.jsonl` with ~300 positives + ~300 hard negatives.


In [ ]:
# [PIPELINE_PATCH_v1] # [PIPELINE_PATCH_v2_center] # [PIPELINE_PATCH_v5_safe_encode]
# v2 consistency-eval builder: removes the `clip_score is not None` filter
# because the center-frame extractor sets clip_score=None for all pairs.
# Also: defensive encode_texts() that handles transformers versions where
# get_text_features() returns BaseModelOutputWithPooling instead of a tensor.
import numpy as np
import torch
from transformers import CLIPModel, CLIPTokenizer

random.seed(2026)
np.random.seed(2026)

# v2: keep every record that has a frame path (no clip_score filter).
eval_pool = [p for p in m2_pairs if p.get("frame_path")]
print(f"Eligible eval pool: {len(eval_pool)}")

N_POS = min(300, len(eval_pool))
positives = random.sample(eval_pool, N_POS)

print("Encoding queries with CLIP-text...")
device   = "cuda" if torch.cuda.is_available() else "cpu"
clip_hf  = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_tok = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")


def _to_text_tensor(result):
    """Coerce get_text_features() output to a 2-D tensor across HF versions."""
    if isinstance(result, torch.Tensor):
        return result
    if hasattr(result, "pooler_output") and result.pooler_output is not None:
        return clip_hf.text_projection(result.pooler_output)
    if hasattr(result, "last_hidden_state"):
        return clip_hf.text_projection(result.last_hidden_state[:, 0])
    raise RuntimeError(f"Unexpected get_text_features output: {type(result)}")


@torch.no_grad()
def encode_texts(texts, batch=64):
    out = []
    for i in range(0, len(texts), batch):
        toks = clip_tok(texts[i:i+batch], padding=True, truncation=True,
                        return_tensors="pt").to(device)
        result = clip_hf.get_text_features(**toks)
        emb    = _to_text_tensor(result)
        emb    = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        out.append(emb.cpu().numpy())
    return np.concatenate(out, axis=0)

queries = [p["query"] for p in eval_pool]
q_emb   = encode_texts(queries)
qid_to_idx = {p["qid"]: i for i, p in enumerate(eval_pool)}

DISSIM_MAX = 0.60   # negative must satisfy cos(q_i, q_j) < this

negatives = []
for p in positives:
    i = qid_to_idx[p["qid"]]
    sims = q_emb @ q_emb[i]
    candidates = [j for j, s in enumerate(sims)
                  if j != i and s < DISSIM_MAX
                  and eval_pool[j]["vid"] != p["vid"]]
    if not candidates:
        continue
    j = random.choice(candidates)
    neg_video = eval_pool[j]
    negatives.append({
        "pair_id"   : f"neg_{p['qid']}_{neg_video['qid']}",
        "label"     : 0,
        "thumb_qid" : p["qid"], "thumb_vid": p["vid"],
        "frame_path": p["frame_path"], "thumb_query": p["query"],
        "video_qid" : neg_video["qid"], "video_vid": neg_video["vid"],
        "video_query_for_ref": neg_video["query"],
        "neg_query_cos": float(sims[j]),
    })

positive_pairs = [{
    "pair_id"   : f"pos_{p['qid']}",
    "label"     : 1,
    "thumb_qid" : p["qid"], "thumb_vid": p["vid"],
    "frame_path": p["frame_path"], "thumb_query": p["query"],
    "video_qid" : p["qid"], "video_vid": p["vid"],
    "video_query_for_ref": p["query"],
} for p in positives]

consistency_eval = positive_pairs + negatives
random.shuffle(consistency_eval)

save_jsonl(consistency_eval, PROCESSED_DIR / "consistency_eval.jsonl")
print(f"\nConsistency eval: {len(positive_pairs)} positives + "
      f"{len(negatives)} hard negatives = {len(consistency_eval)} pairs")
if negatives:
    cosines = [n["neg_query_cos"] for n in negatives]
    print(f"Negative query-cos range: {min(cosines):.3f} .. {max(cosines):.3f}")


# [PIPELINE_PATCH_v1]
# 17. Cache CLIP-image embeddings for all frames + thumbnails

The consistency scorer needs `e_T = CLIP_img(thumbnail)` for every eval pair.
Compute once, cache to `.npy`, avoid re-encoding during the evaluation loop
(saves ~30 minutes when iterating on the score formula).


In [ ]:
# [PIPELINE_PATCH_v1] # [PIPELINE_PATCH_v5_safe_encode] # [PIPELINE_PATCH_v6_bypass_get_features]
# Cache CLIP-image embeddings keyed by qid (m2 frames) and youtube_id (thumbs).
# v6: bypass get_image_features entirely -- some transformers versions have a
# broken visual_projection chain inside get_image_features. We call
# vision_model directly and apply visual_projection only when the dimension
# matches its expected input.

import numpy as np
import torch
from pathlib import Path
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

device    = "cuda" if torch.cuda.is_available() else "cpu"
clip_hf   = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

PROJ_DIM      = clip_hf.config.projection_dim              # 512 for ViT-B/32
VISION_HIDDEN = clip_hf.config.vision_config.hidden_size   # 768 for ViT-B/32
print(f"CLIP config -> projection_dim={PROJ_DIM}, vision_hidden={VISION_HIDDEN}")


@torch.no_grad()
def _vision_pooled(pixel_values):
    """Return post-pooler feature from vision_model, regardless of API shape."""
    out = clip_hf.vision_model(pixel_values=pixel_values)
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output
    if hasattr(out, "last_hidden_state"):
        return out.last_hidden_state[:, 0]
    if isinstance(out, tuple):
        return out[1] if len(out) > 1 else out[0][:, 0]
    raise RuntimeError(f"Unexpected vision_model output: {type(out)}")


@torch.no_grad()
def encode_images(paths, batch=32):
    embs, kept = [], []
    for i in range(0, len(paths), batch):
        bp = paths[i:i+batch]
        imgs, ok = [], []
        for p in bp:
            try:
                imgs.append(Image.open(p).convert("RGB"))
                ok.append(p)
            except Exception:
                continue
        if not imgs:
            continue
        inp    = clip_proc(images=imgs, return_tensors="pt").to(device)
        pooled = _vision_pooled(inp["pixel_values"])

        # Project only if dim matches visual_projection's input.
        if pooled.shape[-1] == VISION_HIDDEN:
            emb = clip_hf.visual_projection(pooled)
        elif pooled.shape[-1] == PROJ_DIM:
            emb = pooled
        else:
            raise RuntimeError(
                f"Pooled feature has unexpected dim {pooled.shape[-1]}; "
                f"expected {VISION_HIDDEN} or {PROJ_DIM}."
            )
        emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        embs.append(emb.cpu().numpy())
        kept.extend(ok)
    if not embs:
        return np.zeros((0, PROJ_DIM), dtype=np.float32), []
    return np.concatenate(embs, axis=0), kept

# 1) Frames (m2 training pairs)
frame_paths = [p["frame_path"] for p in m2_pairs if p.get("frame_path")]
frame_qids  = [p["qid"]        for p in m2_pairs if p.get("frame_path")]
print(f"Encoding {len(frame_paths)} frames...")
frame_emb, kept_paths = encode_images(frame_paths)
kept_set = set(kept_paths)
kept_qids = [q for q, p in zip(frame_qids, frame_paths) if p in kept_set]
np.savez(PROCESSED_DIR / "m2_image_emb.npz",
         qids=np.asarray(kept_qids, dtype=np.int64),
         emb=frame_emb.astype(np.float32))
print(f"  Saved m2_image_emb.npz: {frame_emb.shape}")

# 2) Real thumbnails (one per youtube_id) -- only if step 13 ran
THUMB_DIR = PROJECT_ROOT / "qvh_thumbnails"
if THUMB_DIR.exists():
    thumb_paths = sorted(THUMB_DIR.glob("*.jpg"))
    if thumb_paths:
        print(f"Encoding {len(thumb_paths)} real thumbnails...")
        thumb_emb, kept_thumb = encode_images([str(p) for p in thumb_paths])
        thumb_yt = [Path(p).stem for p in kept_thumb]
        np.savez(PROCESSED_DIR / "thumbnail_image_emb.npz",
                 youtube_ids=np.asarray(thumb_yt),
                 emb=thumb_emb.astype(np.float32))
        print(f"  Saved thumbnail_image_emb.npz: {thumb_emb.shape}")
    else:
        print("(No thumbnails in qvh_thumbnails/ -- skip)")
else:
    print("(qvh_thumbnails/ does not exist -- skip)")


# [PIPELINE_PATCH_v1]
# 18. Final Dataset Manifest

All artifacts produced by this notebook (after the pipeline patch):

| File | Purpose |
|---|---|
| `m1_train_subset.jsonl` | 2000-sample QVH-format subset (for M1 audits / optional retrain) |
| `m1_val.jsonl` | Full official val (for M1 eval) |
| `m2_train_pairs.jsonl` | ~1608 (frame, query) pairs for ClipCap training |
| `m2_val_pairs.jsonl` | ~300 (frame, query) pairs from val (for early stopping) |
| `consistency_eval.jsonl` | ~600 (positive + hard negative) pairs (primary eval) |
| `thumbnail_index.jsonl` | Real-YouTube-thumbnail paths keyed by qid |
| `m2_image_emb.npz` | Cached CLIP-image embeddings for frames |
| `thumbnail_image_emb.npz` | Cached CLIP-image embeddings for real thumbnails |
| `category_map.json` | qid -> category for stratification |
| `download_log.json` | yt-dlp results (debug) |


In [ ]:
# [PIPELINE_PATCH_v1]
# Print final manifest summary.

import os

manifest = [
    "m1_train_subset.jsonl",
    "m1_val.jsonl",
    "m2_train_pairs.jsonl",
    "m2_val_pairs.jsonl",
    "consistency_eval.jsonl",
    "thumbnail_index.jsonl",
    "m2_image_emb.npz",
    "thumbnail_image_emb.npz",
    "category_map.json",
    "download_log.json",
]

print(f"{'File':40s}  {'Status':8s}  {'Size':>12s}")
print("-" * 64)
for name in manifest:
    p = PROCESSED_DIR / name
    if p.exists():
        size = p.stat().st_size
        size_s = f"{size/1024:.1f} KB" if size < 1e6 else f"{size/1e6:.1f} MB"
        print(f"{name:40s}  {'OK':8s}  {size_s:>12s}")
    else:
        print(f"{name:40s}  {'MISSING':8s}  {'-':>12s}")


# [PIPELINE_PATCH_v8_unpack_features]
# 2.5 Unpack Official Moment-DETR Features (one-time, ~5 min)

The M1 baseline + the consistency scorer both need pre-extracted CLIP video and
text features from the official Moment-DETR release. Run this once after section 2.

Source: `moment_detr_features.tar.gz` (place it under `PROJECT_ROOT` first).

Outputs:
- `moment_detr/features/clip_features/*.npz`         -- per-video CLIP features
- `moment_detr/features/clip_text_features/*.npz`    -- per-query CLIP text features


In [ ]:
# # [PIPELINE_PATCH_v8_unpack_features]
# import tarfile

# ARCHIVE  = PROJECT_ROOT / "moment_detr_features.tar.gz"
# FEAT_DIR = REPO_DIR / "features"

# if (FEAT_DIR / "clip_features").exists():
#     print(f"Features already unpacked at {FEAT_DIR}")
# else:
#     if not ARCHIVE.exists():
#         print(f"MISSING: {ARCHIVE}")
#         print("Download the archive from Moment-DETR's release page first:")
#         print("  https://github.com/jayleicn/moment_detr/blob/main/data/README.md")
#     else:
#         print(f"Unpacking {ARCHIVE} -> {REPO_DIR}")
#         with tarfile.open(ARCHIVE) as tf:
#             tf.extractall(REPO_DIR)
#         print(f"Done. Feature dirs: {[p.name for p in FEAT_DIR.iterdir()]}")
#         clip_n = len(list((FEAT_DIR / "clip_features").glob("*.npz")))
#         txt_n  = len(list((FEAT_DIR / "clip_text_features").glob("*.npz")))
#         print(f"  clip_features      : {clip_n} files")
#         print(f"  clip_text_features : {txt_n} files")


# [PIPELINE_PATCH_v10_taxonomy]
# 0.5 — Data Taxonomy (read me first)

Every artifact this notebook produces serves a specific role. Use this map to
understand where each folder/file is consumed downstream.

| Path | Built by | Used by |
|---|---|---|
| `qvh_videos/{vid}.mp4`              | Section 8 (yt-dlp download)               | All frame extractors |
| `qvh_frames/{vid}.jpg`              | Section 9 (center / CLIP-ranked)          | **M2 training (single-frame)**, eval positives |
| `qvh_frames_multi/{vid}_clip{N}.jpg` | v9 multi-frame aug cell                | **M2 training (multi-frame, RECOMMENDED)** |
| `qvh_frames_within_neg/{vid}_clip{N}.jpg` | Hard eval builder (v14)            | **Within-video hard negatives** in `consistency_eval_hard.jsonl` |
| `qvh_thumbnails/{youtube_id}.jpg`   | Section 13 (direct HTTP from i.ytimg.com) | **Optional domain-gap probe**: replace `frame_path` with `thumbnail_path` and re-run eval to measure how much real stylized thumbnails hurt AUC |
| `processed/train_processed.jsonl`     | Section 5                                  | Source for all frame extraction |
| `processed/m1_*.jsonl`                | Section 14                                  | M1 audits / optional retrain |
| `processed/m2_train_pairs.jsonl`      | Section 10                                  | M2 single-frame training (fallback) |
| `processed/m2_train_pairs_multi.jsonl`| v9 multi-frame aug                          | **★ Default M2 training set** |
| `processed/m2_val_pairs.jsonl`        | v7 self-contained M2-val builder            | M2 early stopping (CLIP-text-sim) |
| `processed/consistency_eval.jsonl`    | Section 16                                  | **E3 ceiling** AUC (cross-video) |
| `processed/consistency_eval_hard.jsonl` | v14 hard builder                          | **E4 research claim** AUC (within-video) |
| `processed/consistency_eval_query_mismatch.jsonl` | v14 qmiss builder               | **E5 alpha-sweep** (query path test) |
| `processed/manual_gold.jsonl`         | v12 gold builder + you labeling             | **E6 small-but-real** AUC (~50 pairs) |
| `processed/m2_image_emb.npz`          | Section 17                                  | Cached frame embeddings (faster eval) |
| `processed/thumbnail_image_emb.npz`   | Section 17                                  | Cached thumbnail embeddings (for domain-gap probe) |
| `processed/category_map.json`         | Section 6                                   | Stratification + reporting |

## What is the QV thumbnails section for?

You probably wondered: we downloaded `qvh_thumbnails/`, but the training pipeline
never uses them. That's intentional. **They are an evaluation-only artifact for
the domain-gap experiment** (E8 in EVALUATION.md):

1. M2 was trained on **video frames** (CLIP-similar, "natural" images).
2. In a real deployment, M2 receives **stylized YouTube thumbnails** (text
   overlays, color grading, edited composites).
3. The gap between these two distributions degrades M2's queries.
4. To **measure** this gap, re-run the consistency scorer using the real
   thumbnails as the `thumbnail_path` (instead of `qvh_frames/{vid}.jpg`)
   and compare AUC.

If you skip the domain-gap experiment, you can ignore `qvh_thumbnails/`
entirely — it doesn't affect training or the main AUC.


In [ ]:
# [PIPELINE_PATCH_v11_reconcile]
# Reconcile previously-extracted v1 frames with the current pipeline.
#
# Use this cell IF: you already extracted frames with the old v1 (CLIP-ranked)
# method and want to keep them. This cell DOES NOT re-extract anything --
# it just rebuilds `m2_pairs` in memory from whatever .jpg files exist in
# FRAMES_DIR, so downstream cells use them.
#
# Skip this cell if you ran the v2/v3 center-frame extractor in this same
# session (m2_pairs is already correct).

import json
from pathlib import Path


def _load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]


def _save_jsonl(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


# Reload train_subset from disk if not in memory.
if 'train_subset' not in dir():
    tp = PROCESSED_DIR / 'train_processed.jsonl'
    assert tp.exists(), f"Missing {tp} -- run Sections 5-7 first."
    train_subset = _load_jsonl(tp)

# Mark frame_path on each record whose .jpg exists on disk.
n_with_frame = 0
for r in train_subset:
    candidate = FRAMES_DIR / f"{r['vid']}.jpg"
    if candidate.exists():
        r['frame_path']       = str(candidate)
        r['video_downloaded'] = True
        if 'clip_score' not in r:
            r['clip_score'] = None
        n_with_frame += 1
    else:
        r['frame_path']       = None
        r['video_downloaded'] = False

# Rebuild m2_pairs from records that have frames.
m2_pairs = [{
    'qid'           : r['qid'],
    'frame_path'    : r['frame_path'],
    'query'         : r['query'],
    'vid'           : r['vid'],
    'best_timestamp': r.get('best_timestamp'),
    'mean_saliency' : r.get('mean_saliency'),
    'clip_score'    : r.get('clip_score'),
} for r in train_subset if r.get('frame_path')]

_save_jsonl(train_subset, PROCESSED_DIR / 'train_processed.jsonl')
_save_jsonl(m2_pairs,     PROCESSED_DIR / 'm2_train_pairs.jsonl')

print(f"train_subset           : {len(train_subset):,} records")
print(f"records with frame     : {n_with_frame:,}")
print(f"m2_pairs (frame, query): {len(m2_pairs):,}")
print(f"\nIf m2_pairs > 0, downstream cells (multi-frame, eval builders) are ready to run.")


# [PIPELINE_PATCH_v14_hard_qmiss]
# 16.5 — Build the harder evaluation sets

Three eval sets, in increasing difficulty:

1. `consistency_eval.jsonl`              -- cross-video negatives (already built in Section 16)
2. `consistency_eval_hard.jsonl`         -- ★ within-video negatives (built below) — the research claim
3. `consistency_eval_query_mismatch.jsonl` -- query-mismatch negatives (built below) — isolates the query path

Each has 300 positives + 300 negatives = 600 pairs total.


In [ ]:
# [PIPELINE_PATCH_v14_hard_qmiss]
# Build consistency_eval_hard.jsonl -- within-video negatives.
# For each positive, pick a frame from OUTSIDE the relevant_window of the SAME video.

import cv2, random, json
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

random.seed(2026)

WITHIN_NEG_DIR = PROJECT_ROOT / 'qvh_frames_within_neg'
WITHIN_NEG_DIR.mkdir(parents=True, exist_ok=True)

# Load the existing positives + train records
positives = [p for p in m2_pairs if p.get('frame_path')]
train_by_qid = {r['qid']: r for r in train_subset}

def extract_outside_frame(record):
    n_clips = int(record.get('duration', 150) // 2) if record.get('duration') else 75
    relevant = set(record.get('relevant_clip_ids', []))
    outside = [c for c in range(n_clips) if c not in relevant]
    if not outside:
        return None
    # Prefer clips at distance >= 5 from any relevant clip (avoid transitions)
    if relevant:
        far = [c for c in outside if min(abs(c - r) for r in relevant) >= 5]
        outside = far if far else outside
    chosen = random.choice(outside)
    video_path = VIDEO_DIR / f"{record['vid']}.mp4"
    if not video_path.exists():
        return None
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    cap.set(cv2.CAP_PROP_POS_MSEC, float(chosen * 2_000 + 1_000))
    ret, fr = cap.read()
    cap.release()
    if not ret:
        return None
    out_path = WITHIN_NEG_DIR / f"{record['vid']}_clip{chosen}.jpg"
    pil = Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)).resize((224, 224), Image.LANCZOS)
    pil.save(str(out_path), quality=92)
    return {'path': str(out_path), 'clip_id': chosen}

# Pick 300 positives for the hard eval (a balanced subset)
N = min(300, len(positives))
chosen = random.sample(positives, N)

hard_eval = []
for p in tqdm(chosen, desc='within-video negatives'):
    rec = train_by_qid.get(p['qid'])
    if rec is None or not rec.get('relevant_clip_ids'):
        continue
    # Positive
    hard_eval.append({
        'pair_id'    : f"pos_{p['qid']}",
        'label'      : 1,
        'thumb_qid'  : p['qid'], 'thumb_vid': p['vid'],
        'frame_path' : p['frame_path'],
        'thumb_query': p['query'],
        'video_qid'  : p['qid'], 'video_vid': p['vid'],
    })
    # Within-video negative
    neg = extract_outside_frame(rec)
    if neg is None:
        continue
    hard_eval.append({
        'pair_id'    : f"hardneg_{p['qid']}_clip{neg['clip_id']}",
        'label'      : 0,
        'thumb_qid'  : p['qid'], 'thumb_vid': p['vid'],
        'frame_path' : neg['path'],
        'thumb_query': p['query'],
        'video_qid'  : p['qid'], 'video_vid': p['vid'],
        'out_of_window_clip_id': neg['clip_id'],
    })

random.shuffle(hard_eval)
out_path = PROCESSED_DIR / 'consistency_eval_hard.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for x in hard_eval:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')

print(f"Hard eval written: {out_path}")
print(f"  positives        : {sum(1 for p in hard_eval if p['label']==1)}")
print(f"  within-video neg : {sum(1 for p in hard_eval if p['label']==0)}")


In [ ]:
# [PIPELINE_PATCH_v14_hard_qmiss]
# Build consistency_eval_query_mismatch.jsonl -- query-mismatch negatives.
# Same frame + same video, but a WRONG query (semantically unrelated).

import numpy as np
import torch
from transformers import CLIPModel, CLIPTokenizer

random.seed(2026)
np.random.seed(2026)

positives = [p for p in m2_pairs if p.get('frame_path')]
print(f"Encoding {len(positives)} queries with CLIP-text...")
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_hf = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_tok = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

@torch.no_grad()
def encode_texts(texts, batch=64):
    out = []
    for i in range(0, len(texts), batch):
        toks = clip_tok(texts[i:i+batch], padding=True, truncation=True,
                        return_tensors="pt").to(device)
        result = clip_hf.text_model(**toks).pooler_output
        emb = clip_hf.text_projection(result)
        emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        out.append(emb.cpu().numpy())
    return np.concatenate(out, axis=0)

all_queries = [p['query'] for p in positives]
q_emb = encode_texts(all_queries)

DISSIM_MAX = 0.40
qmiss_eval = []
N = min(300, len(positives))
chosen = random.sample(range(len(positives)), N)

for i in chosen:
    p = positives[i]
    sims = q_emb @ q_emb[i]
    cands = [j for j, s in enumerate(sims) if j != i and s < DISSIM_MAX]
    if not cands:
        continue
    j = random.choice(cands)
    wrong_query = all_queries[j]

    qmiss_eval.append({
        'pair_id'    : f"pos_{p['qid']}",
        'label'      : 1,
        'thumb_qid'  : p['qid'], 'thumb_vid': p['vid'],
        'frame_path' : p['frame_path'],
        'thumb_query': p['query'],
        'video_qid'  : p['qid'], 'video_vid': p['vid'],
    })
    qmiss_eval.append({
        'pair_id'           : f"qmiss_{p['qid']}_with_q{positives[j]['qid']}",
        'label'             : 0,
        'thumb_qid'         : p['qid'], 'thumb_vid': p['vid'],
        'frame_path'        : p['frame_path'],
        'thumb_query'       : wrong_query,
        'video_qid'         : p['qid'], 'video_vid': p['vid'],
        'correct_query_was' : p['query'],
        'wrong_query_from'  : positives[j]['qid'],
        'cos_correct_wrong' : float(sims[j]),
    })

random.shuffle(qmiss_eval)
out_path = PROCESSED_DIR / 'consistency_eval_query_mismatch.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for x in qmiss_eval:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')

print(f"Query-mismatch eval: {out_path}")
print(f"  positives        : {sum(1 for p in qmiss_eval if p['label']==1)}")
print(f"  query-mismatch neg: {sum(1 for p in qmiss_eval if p['label']==0)}")


# [PIPELINE_PATCH_v12_gold_set]
# 19 — Manual Gold Evaluation Set (the "ground truth" sanity check)

The synthetic eval sets (cross-video, within-video, query-mismatch) are built
**automatically** from QVH structure. They give us 600 pairs with zero labeling
effort -- but they may not reflect real human judgment of consistency.

The **manual gold set** is a small sample of 50 pairs that team members label
by hand. We then compute AUC on this set and compare it against the synthetic
AUC. If they're close, the synthetic eval is a faithful proxy.

This section produces:
1. `manual_gold_template.jsonl` -- 50 pairs to label, with `label_human=null`
2. `manual_gold.jsonl` -- the SAME file after each team member fills in labels

**Time investment**: ~2 hours total across 4 people (12-13 pairs each).


In [ ]:
# [PIPELINE_PATCH_v12_gold_set]
# Build the gold template: 50 pairs (25 positives + 25 negatives) sampled from
# consistency_eval.jsonl, with empty label_human fields to fill manually.

import random, json
from pathlib import Path

random.seed(2026)

CONSIST_PATH = PROCESSED_DIR / 'consistency_eval.jsonl'
assert CONSIST_PATH.exists(), f"Run Section 16 first to build {CONSIST_PATH}"

pairs = []
with open(CONSIST_PATH, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            pairs.append(json.loads(line))

pos = [p for p in pairs if p['label'] == 1]
neg = [p for p in pairs if p['label'] == 0]

# Stratified sample: 25 of each
N_PER_LABEL = 25
sampled = random.sample(pos, min(N_PER_LABEL, len(pos))) \
        + random.sample(neg, min(N_PER_LABEL, len(neg)))
random.shuffle(sampled)

# Build template with empty label_human
template = []
for p in sampled:
    template.append({
        'pair_id'           : p['pair_id'],
        'label_synthetic'   : p['label'],
        'label_human'       : None,           # fill: 0=inconsistent, 1=consistent, 0.5=partial
        'confidence'        : None,           # fill: 1=sure, 2=somewhat sure, 3=guessing
        'note'              : '',              # fill: optional comment
        'thumb_vid'         : p['thumb_vid'],
        'frame_path'        : p['frame_path'],
        'thumb_query'       : p['thumb_query'],
        'video_vid'         : p['video_vid'],
        'video_query_for_ref': p.get('video_query_for_ref'),
    })

out_path = PROCESSED_DIR / 'manual_gold_template.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for r in template:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f"Wrote template: {out_path}")
print(f"  total pairs    : {len(template)}")
print(f"  synthetic pos  : {sum(1 for r in template if r['label_synthetic']==1)}")
print(f"  synthetic neg  : {sum(1 for r in template if r['label_synthetic']==0)}")
print(f"\nSee docs/LABELING_GUIDE.md for instructions on how to fill labels.")
print(f"Save the labeled file as: {PROCESSED_DIR}/manual_gold.jsonl")


In [ ]:
# [PIPELINE_PATCH_v12_gold_set]
# Interactive viewer: display each gold-set pair (thumb + video frames + query).
# Use this to label by eye. Records can be filled in the template JSONL using
# any editor.

import cv2, json
from PIL import Image
import matplotlib.pyplot as plt
import textwrap
from pathlib import Path

template_path = PROCESSED_DIR / 'manual_gold_template.jsonl'
if not template_path.exists():
    print(f"Run the previous cell first to build {template_path}")
else:
    pairs = []
    with open(template_path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                pairs.append(json.loads(line))


    def show_pair(p, n_video_frames=4):
        fig, axes = plt.subplots(1, 1 + n_video_frames, figsize=(3.5 * (1 + n_video_frames), 3.5))
        # Thumbnail
        try:
            axes[0].imshow(Image.open(p['frame_path']).convert('RGB'))
            axes[0].set_title(f"THUMB\n{p['thumb_vid'][:22]}", fontsize=9)
        except Exception:
            axes[0].text(0.5, 0.5, 'thumb missing', ha='center')
        axes[0].axis('off')

        # Video frames sampled from the .mp4
        vp = VIDEO_DIR / f"{p['video_vid']}.mp4"
        if vp.exists():
            cap = cv2.VideoCapture(str(vp))
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
            for j in range(n_video_frames):
                target = int(total * (j + 0.5) / n_video_frames) if total else 0
                cap.set(cv2.CAP_PROP_POS_FRAMES, target)
                ret, fr = cap.read()
                ax = axes[1 + j]
                if ret:
                    img = Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
                    ax.imshow(img); ax.set_title(f"VIDEO @ ~{int(100*(j+0.5)/n_video_frames)}%", fontsize=9)
                else:
                    ax.text(0.5, 0.5, 'N/A', ha='center')
                ax.axis('off')
            cap.release()
        else:
            for j in range(n_video_frames):
                axes[1 + j].text(0.5, 0.5, 'no .mp4', ha='center'); axes[1 + j].axis('off')

        qtext = textwrap.fill(p['thumb_query'], 100)
        synth = 'POSITIVE' if p['label_synthetic'] == 1 else 'NEGATIVE'
        ref = p.get('video_query_for_ref') or '(same as thumb_query for positives)'
        fig.suptitle(
            f"pair_id={p['pair_id']}  |  synthetic_label={synth}\n"
            f"Thumb's query : {qtext}\n"
            f"Video's actual: {textwrap.fill(ref, 90)}",
            fontsize=10, y=1.05)
        plt.tight_layout(); plt.show()


    # Show the first 5 pairs as a demo. Re-run to see more, or batch them all.
    for p in pairs[:5]:
        show_pair(p)
        print(f"  pair_id  : {p['pair_id']}")
        print(f"  TO LABEL : edit manual_gold_template.jsonl, set:")
        print(f"             label_human = 0 (inconsistent) or 1 (consistent) or 0.5 (partial)")
        print(f"             confidence  = 1, 2, or 3")
        print('-' * 70)


# [PIPELINE_PATCH_v13_gold_eval]
# 20 — Gold-set pipeline test (run after labeling is complete)

After all 50 pairs have `label_human` filled in (saved as `manual_gold.jsonl`),
this cell runs the consistency scorer on them and computes:

- **Gold AUC**: AUC using `label_human` (binarize 0.5 -> 0)
- **Synthetic AUC**: AUC using `label_synthetic`
- **Inter-rater agreement** if multiple labelers contributed
- **Synthetic vs gold correlation**: does the synthetic eval reflect human judgment?

If `gold AUC` ≈ `synthetic AUC` (within ~0.05), the synthetic eval is a faithful
proxy and we can confidently report the 600-pair synthetic numbers in the paper.


In [ ]:
# [PIPELINE_PATCH_v13_gold_eval]
# Run pipeline on the labeled gold set and compute AUC.
# Requires: src.scoring.* modules; bootstrap cell (in modular notebooks) sets sys.path.
# For data_collection_preparation.ipynb, ensure REPO_DIR is on sys.path first.

import sys, json
from pathlib import Path

REPO_DIR = '/content/drive/MyDrive/thumbnail_project/thumbnail-consistency'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np

GOLD_PATH = PROCESSED_DIR / 'manual_gold.jsonl'
if not GOLD_PATH.exists():
    print(f"Gold file missing: {GOLD_PATH}")
    print("Label the template first (see docs/LABELING_GUIDE.md), then save")
    print("the result here as 'manual_gold.jsonl'.")
else:
    gold = []
    with open(GOLD_PATH, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                gold.append(json.loads(line))

    # Filter records that have a human label
    labeled = [r for r in gold if r.get('label_human') is not None]
    print(f"Total gold pairs   : {len(gold)}")
    print(f"Labeled (human)    : {len(labeled)}")
    if not labeled:
        print("No human labels filled yet. Stop here.")
    else:
        # Convert 0.5 (partial) -> 0 for AUC binarization
        h_lab = np.array([1 if r['label_human'] == 1 else 0 for r in labeled])
        s_lab = np.array([r['label_synthetic'] for r in labeled])

        # Agreement
        agree = (h_lab == s_lab).mean()
        print(f"\nHuman vs synthetic label agreement: {agree:.3f}")
        if agree < 0.75:
            print("  WARNING: low agreement. Synthetic eval may not be a good proxy.")

        # Show breakdown
        from collections import Counter
        print(f"\nHuman label distribution    : {Counter(h_lab.tolist())}")
        print(f"Synthetic label distribution: {Counter(s_lab.tolist())}")

        # Now run the scorer (requires src/* on sys.path + features extracted)
        # If you haven't run the smoke test yet, you can compute scores here too.
        # For now we just report the agreement number and let you compute AUC
        # in the run_evaluation.ipynb notebook with EVAL_PATH = manual_gold.jsonl.

        print(f"\nNext step: open notebooks/run_evaluation.ipynb and change")
        print(f"  EVAL_PATH = '/content/drive/.../processed/manual_gold.jsonl'")
        print(f"Then re-run the scoring loop and compare AUC vs the synthetic AUC.")


# [PIPELINE_PATCH_v9_multiframe_aug]
# 10.5 — Multi-frame augmentation for M2 training

For each annotated record we extract **K=3 frames** evenly spaced across the
`relevant_clip_ids` (the moment) and treat each `(frame, query)` as a separate
training example. This produces ~3× the training data of the single-frame set
and gives M2 more visual diversity per query.

Output: `processed/m2_train_pairs_multi.jsonl` -- the **★ recommended training
set** for ClipCap (M2). The single-frame `m2_train_pairs.jsonl` stays as a
fallback.

Frames land in `qvh_frames_multi/{vid}_clip{N}.jpg` (a separate folder from
`qvh_frames/`, so no file collisions).


In [ ]:
# [PIPELINE_PATCH_v9_multiframe_aug]
# === Multi-frame augmentation for M2 training (full QVH set) ===
# Treats each (frame, query) as a separate training example.
# Scales with however many records are in train_processed.jsonl.

import cv2, random, json
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

random.seed(2026)

# CONFIG
K_FRAMES_PER_QUERY = 3
MIN_RELEVANT_CLIPS = 1     # skip records with no relevant clips annotated
QVH_VIDEOS         = VIDEO_DIR
QVH_MULTI_FRAMES   = PROJECT_ROOT / 'qvh_frames_multi'
QVH_MULTI_FRAMES.mkdir(parents=True, exist_ok=True)

# Load all train records that have annotation
TRAIN_PROC = PROCESSED_DIR / 'train_processed.jsonl'
train_records = []
with open(TRAIN_PROC, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        if r.get('relevant_clip_ids') and len(r['relevant_clip_ids']) >= MIN_RELEVANT_CLIPS:
            train_records.append(r)

print(f"Loaded {len(train_records):,} train records with annotation")
print(f"Will extract K={K_FRAMES_PER_QUERY} frames per record")
print(f"Expected output: ~{K_FRAMES_PER_QUERY * len(train_records):,} (frame, query) pairs")


def sample_relevant_frames(record, k=3):
    """Extract k frames spread across the relevant_clip_ids of a record."""
    relevant = sorted(record['relevant_clip_ids'])
    video_path = QVH_VIDEOS / f"{record['vid']}.mp4"
    if not video_path.exists():
        return []

    # Spread k indices across the relevant clip range for visual diversity
    if len(relevant) >= k:
        idxs = [int(round((len(relevant) - 1) * i / max(1, k - 1))) for i in range(k)]
        chosen = [relevant[i] for i in idxs]
    else:
        chosen = relevant + random.choices(relevant, k=k - len(relevant))

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    out = []
    for clip_id in chosen:
        out_path = QVH_MULTI_FRAMES / f"{record['vid']}_clip{clip_id}.jpg"
        if out_path.exists():
            out.append({'path': str(out_path), 'clip_id': clip_id})
            continue
        cap.set(cv2.CAP_PROP_POS_MSEC, float(clip_id * 2_000 + 1_000))
        ret, fr = cap.read()
        if not ret:
            continue
        pil = (Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
                   .resize((224, 224), Image.LANCZOS))
        pil.save(str(out_path), quality=92)
        out.append({'path': str(out_path), 'clip_id': clip_id})
    cap.release()
    return out


m2_pairs_multi = []
n_missing_video = 0
for rec in tqdm(train_records, desc='Multi-frame extraction'):
    frames = sample_relevant_frames(rec, k=K_FRAMES_PER_QUERY)
    if not frames:
        n_missing_video += 1
        continue
    for f in frames:
        m2_pairs_multi.append({
            'qid'           : rec['qid'],
            'vid'           : rec['vid'],
            'frame_path'    : f['path'],
            'query'         : rec['query'],
            'clip_id'       : f['clip_id'],
            'best_timestamp': f['clip_id'] * 2 + 1.0,
            'mean_saliency' : rec.get('mean_saliency'),
            'augment_id'    : f"{rec['vid']}_clip{f['clip_id']}",
        })

out_path = PROCESSED_DIR / 'm2_train_pairs_multi.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for p in m2_pairs_multi:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

print(f"\nWrote {out_path}")
print(f"  records with frames extracted : {len(train_records) - n_missing_video:,}")
print(f"  records skipped (missing video): {n_missing_video:,}")
print(f"  total (frame, query) pairs    : {len(m2_pairs_multi):,}")
